<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/TweetsGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,Dataset
from transformers import AutoModelForCausalLM,AutoModel,AutoTokenizer
import os
from google.colab import userdata
from sklearn.model_selection import train_test_split
import pandas as pd
from tqdm import tqdm
import gc

In [ ]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
!kaggle datasets download -d gpreda/bbc-news

In [ ]:
!unzip -q bbc-news.zip -d ./datafolder/

In [ ]:
df = pd.read_csv('/content/datafolder/bbc_news.csv')

In [ ]:
df.isnull().sum()

In [ ]:
# df = df.dropna(subset=['statement'])

In [ ]:
# df = df[df['statement'].str.strip() != ""]

In [ ]:
autoToken = AutoTokenizer.from_pretrained('gpt2')
# 2. THE CRITICAL FIX: GPT-2 needs a padding token
# We tell it to use the 'End of String' token as padding
autoToken.pad_token = autoToken.eos_token

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
max_len = 256
vocab_size = autoToken.vocab_size

In [ ]:
statement_text = df['description'].astype(str).values

In [ ]:
x_train,x_test = train_test_split(statement_text,test_size=0.3,random_state=42)

In [ ]:
train_data = autoToken(text=list(x_train),padding='max_length',max_length=max_len+1,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')
val_data = autoToken(text=list(x_test),padding='max_length',max_length=max_len+1,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

In [ ]:
class SentimentDataset(Dataset):
  def __init__(self,encoding):
    self.encoding = encoding

  def __len__(self):
    return len(self.encoding['input_ids'])

  def __getitem__(self,idx):
    ids = self.encoding['input_ids'][idx]
    mask = self.encoding['attention_mask'][idx]

    return {
        'input_ids':ids[:-1],
        'target_ids':ids[1:],
        'attention_mask':mask[:-1]
    }

In [ ]:
train_d = SentimentDataset(encoding=train_data)
val_d = SentimentDataset(encoding=val_data)

In [ ]:
train_ds = DataLoader(dataset=train_d,batch_size=32,shuffle=True,pin_memory=True,num_workers=2)
val_ds = DataLoader(dataset=val_d,batch_size=32,shuffle=False,pin_memory=True,num_workers=2)

GPT-PreTrained

In [ ]:
# class PreTrainedGPT(nn.Module):
#   def __init__(self) -> None:
#     super().__init__()
#     #Load the model
#     self.gpt2 = AutoModelForCausalLM.from_pretrained('gpt2')

#   def forward(self,input_ids,attention_mask,labels=None):
#     # GPT-2 calculates the loss internally if we pass labels
#     outputs = self.gpt2(input_ids=input_ids,attention_mask=attention_mask,labels=labels)
#     return outputs.loss,outputs.logits

In [ ]:
# model = PreTrainedGPT().to(device)
# optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

In [ ]:
# import torch

# # 1. Initialize the Gradient Scaler before the epoch loop starts
# scaler = torch.cuda.amp.GradScaler()

# epochs = 4

# for epoch in range(epochs):
#     # ==================== TRAINING PHASE ====================
#     model.train()
#     train_loss = 0

#     progress_bar_train = tqdm(train_ds, desc=f"Epoch {epoch+1}")
#     for batch in progress_bar_train:
#         # Unpack tensors and strip extra nested dimensions
#         ids = batch['input_ids'].to(device).squeeze(1)
#         mask = batch['attention_mask'].to(device).squeeze(1)
#         targets = batch['target_ids'].to(device).squeeze(1)

#         optimizer.zero_grad()

#         # Enclose the forward pass in autocast for low-memory 16-bit math
#         with torch.cuda.amp.autocast():
#             loss, logits = model(ids, mask, labels=targets)

#         # Scale the loss and execute backward / step operations safely
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()

#         train_loss += loss.item()
#         progress_bar_train.set_postfix(loss=loss.item())

#     # Calculate and report training metrics OUTSIDE the batch loop
#     avg_train_loss = train_loss / len(train_ds)
#     train_perplexity = torch.exp(torch.tensor(avg_train_loss))
#     print(f"\nTrain Loss: {avg_train_loss:.4f} | Train Perplexity: {train_perplexity:.2f}")

#     # ==================== VALIDATION PHASE ====================
#     model.eval()
#     total_val_loss = 0

#     progress_bar_val = tqdm(val_ds, desc='Validation')
#     with torch.no_grad():
#         for batch in progress_bar_val:
#             # Fixed key name bug: changed 'labels' to your dataset's 'target_ids'
#             v_ids = batch['input_ids'].to(device).squeeze(1)
#             v_mask = batch['attention_mask'].to(device).squeeze(1)
#             v_targets = batch['target_ids'].to(device).squeeze(1)

#             # Use autocast in validation too to save memory during gathering
#             with torch.cuda.amp.autocast():
#                 outputs = model(input_ids=v_ids, attention_mask=v_mask, labels=v_targets)

#             # outputs[0] is the pre-computed loss from PreTrainedGPT
#             total_val_loss += outputs[0].item()
#             progress_bar_val.set_postfix(loss=outputs[0].item())

#     # Calculate and report validation metrics OUTSIDE the batch loop
#     avg_loss = total_val_loss / len(val_ds)
#     val_perplexity = torch.exp(torch.tensor(avg_loss))
#     print(f"Epoch {epoch+1} | Val Loss: {avg_loss:.4f} | Val Perplexity: {val_perplexity:.2f}")
#     print("="*60)


In [ ]:
class CustomGPT2Writer(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # 1. The Pre-trained 'Writer' Brain (768 features)
        self.gpt2 = AutoModel.from_pretrained('gpt2')

        # 2. YOUR OWN CUSTOM LAYERS (Just like you wanted!)
        self.dropout = nn.Dropout(0.2)
        # For a writer, the output MUST be the vocab_size
        self.output_layer = nn.Linear(768, vocab_size)

    def forward(self, input_ids, attention_mask):
        # 1. Get features from the brain
        outputs = self.gpt2(input_ids=input_ids, attention_mask=attention_mask)

        # 2. Grab the hidden states (The words' meaning)
        # Shape: [Batch, Seq_Len, 768]
        x = outputs.last_hidden_state

        # 3. Apply your own custom logic
        x = self.dropout(x)

        # 4. Pass through your own custom output layer
        # Result: [Batch, Seq_Len, Vocab_Size]
        logits = self.output_layer(x)

        return logits

In [ ]:
# Use your custom class that outputs raw logits [prev]
model = CustomGPT2Writer(vocab_size=autoToken.vocab_size)

# Wrap it in DataParallel for your multi-GPU setups
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()
scaler = torch.cuda.amp.GradScaler()

In [ ]:
epochs = 4

# ==================== 2. MAIN CORRECTIONS LOOP ====================
for epoch in range(epochs):

    # -------------------- TRAINING PHASE --------------------
    model.train()
    train_loss = 0

    progress_bar_train = tqdm(train_ds, desc=f"Epoch {epoch+1}")
    for batch in progress_bar_train:
        # Unpack, migrate to GPU, and drop hidden dimensions [prev]
        ids = batch['input_ids'].to(device).squeeze(1)
        mask = batch['attention_mask'].to(device).squeeze(1)
        targets = batch['target_ids'].to(device).squeeze(1)

        optimizer.zero_grad()

        # Enclose manual calculations inside low-memory 16-bit autocast [prev]
        with torch.cuda.amp.autocast():
            # Returns raw hidden logits [Batch, Seq_Len, Vocab_Size] [prev]
            logits = model(ids, mask)

            # Manual 3D -> 2D flattening outside parallelized blocks [prev]
            loss = criterion(logits.view(-1, logits.size(-1)), targets.view(-1))

        # Execute backward and step operations via the gradient scaler [prev]
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        progress_bar_train.set_postfix(loss=loss.item())

    # Calculate metrics for the whole training epoch [prev]
    avg_train_loss = train_loss / len(train_ds)
    train_perplexity = torch.exp(torch.tensor(avg_train_loss))
    print(f"\nTrain Loss: {avg_train_loss:.4f} | Train Perplexity: {train_perplexity:.2f}")

    # ==================== VALIDATION PHASE ====================
    model.eval()
    total_val_loss = 0

    progress_bar_val = tqdm(val_ds, desc='Validation')
    with torch.no_grad():
        for batch in progress_bar_val:
            v_ids = batch['input_ids'].to(device).squeeze(1)
            v_mask = batch['attention_mask'].to(device).squeeze(1)
            v_targets = batch['target_ids'].to(device).squeeze(1)

            with torch.cuda.amp.autocast():
                v_logits = model(v_ids, v_mask)
                v_loss = criterion(v_logits.view(-1, v_logits.size(-1)), v_targets.view(-1))

            # PRO TIP: Pulling .item() breaks the tensor connection
            # and prevents memory leakage into the next epoch
            total_val_loss += v_loss.item()
            progress_bar_val.set_postfix(loss=v_loss.item())

    # Calculate metrics outside the loop
    avg_val_loss = total_val_loss / len(val_ds)
    val_perplexity = torch.exp(torch.tensor(avg_val_loss))
    print(f"Epoch {epoch+1} | Val Loss: {avg_val_loss:.4f} | Val Perplexity: {val_perplexity:.2f}")
    print("=" * 60)

    # 💥 THE CRITICAL MEMORY FLUSH ADDITION
    # This force-cleans the validation tensor cache lines before Epoch 2 starts!
    gc.collect()
    torch.cuda.empty_cache()